In [99]:
import importlib
import torch
import model_def
import numpy as np
importlib.reload(model_def)
from torch import nn
from pathlib import Path
from model_def import get_model
from test_mnist import download, load_images, load_labels
from emnist import extract_training_samples

In [86]:
def load_mnist_train(root="./data"):
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)

    image_file = "train-images-idx3-ubyte.gz"
    label_file = "train-labels-idx1-ubyte.gz"
    image_path = root / image_file
    label_path = root / label_file

    mnist_url = "https://storage.googleapis.com/cvdf-datasets/mnist/"
    download(mnist_url + image_file, image_path)
    download(mnist_url + label_file, label_path)

    images = load_images(image_path)
    labels = load_labels(label_path)

    images = images[:, None, :, :]
    images = images.repeat(3, axis=1)

    return images, labels

In [104]:
def load_emnist_train():
    emnist_images, emnist_labels = extract_training_samples('digits')
    emnist_images = emnist_images.astype(np.float32) / 255.0
    emnist_images = emnist_images[:, None, :, :]
    emnist_images = emnist_images.repeat(3, axis=1)
    return emnist_images, emnist_labels

In [106]:
# emnist_images, emnist_labels = load_emnist_train()
# import matplotlib.pyplot as plt
# for i in [0, 1, 2, 3, 4]:
#     print(f"index {i}, label says: {emnist_labels[i]}")
#     plt.imshow(emnist_images[i, 0], cmap='gray')
#     plt.title(f"label: {emnist_labels[i]}")
#     plt.show()

In [19]:

#### THIS TRAINING IS OUT OF DATA AND NEEDS A DIFF LOSS FUNCTION
def train():
    images, labels = load_mnist_train()
    x = torch.tensor(images, dtype=torch.float32)
    y = torch.tensor(labels, dtype=torch.long)

    model = get_model()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()

    batch_size = 64
    n = x.shape[0]

    for epoch in range(10):
        perm = torch.randperm(n)          # shuffle indices each epoch
        total_loss = 0.0
        for i in range(0, n, batch_size):
            idx = perm[i:i+batch_size]
            xb, yb = x[idx], y[idx]

            optimizer.zero_grad()
            out = model(xb)
            loss = loss_fn(out, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"epoch {epoch}, avg loss {total_loss / (n / batch_size):.4f}")

    torch.save(model.state_dict(), "trained.pt")
    print("saved trained.pt")
train()

epoch 0, avg loss 1.5622
epoch 1, avg loss 1.4968
epoch 2, avg loss 1.4879
epoch 3, avg loss 1.4848
epoch 4, avg loss 1.4815
epoch 5, avg loss 1.4798
epoch 6, avg loss 1.4778
epoch 7, avg loss 1.4759
epoch 8, avg loss 1.4749
epoch 9, avg loss 1.4739
saved trained.pt


In [15]:
#let's add color
def random_tint(images):
    n = images.shape[0]
    tint = torch.rand(n,3,1,1)*0.8+0.2
    return images*tint

def random_fg_bg_tint(images):
    n = images.shape[0]
    fg_color = torch.rand(n, 3, 1, 1)*0.8+0.2
    bg_color = torch.rand(n, 3, 1, 1)*0.8+0.2
    mask = images[ :, 0:1, :, :]
    colored_images = mask*fg_color + (1-mask)*bg_color
    return colored_images

In [107]:
def train_with_color():
    images, labels = load_mnist_train()
    emnist_images, emnist_labels = load_emnist_train()
    images = np.concatenate([images, emnist_images])
    labels = np.concatenate([labels, emnist_labels])

    x = torch.tensor(images, dtype=torch.float32)
    y = torch.tensor(labels, dtype=torch.long)

    model = get_model()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.NLLLoss()

    batch_size = 64
    n = x.shape[0]
    for epoch in range(20):
        perm = torch.randperm(n)
        total_loss = 0.0
        for i in range(0, n, batch_size):
            idx = perm[i:i+batch_size]
            xb, yb = x[idx], y[idx]
            choice = torch.randint(0,3,(1,))
            if choice == 0:
                pass
            elif choice == 1:
                xb = random_tint(xb)
            elif choice == 2:
                xb = random_fg_bg_tint(xb)

            optimizer.zero_grad()
            out = model(xb)
            loss = loss_fn(torch.log(out + 1e-8), yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"epoch {epoch}, avg loss {total_loss / (n / batch_size):.4f}")

    torch.save(model.state_dict(), "trained.pt")
    print("saved trained.pt")
train_with_color()

epoch 0, avg loss 0.3006
epoch 1, avg loss 0.1095
epoch 2, avg loss 0.0880
epoch 3, avg loss 0.0784
epoch 4, avg loss 0.0715
epoch 5, avg loss 0.0652
epoch 6, avg loss 0.0607
epoch 7, avg loss 0.0564
epoch 8, avg loss 0.0552
epoch 9, avg loss 0.0522
epoch 10, avg loss 0.0504
epoch 11, avg loss 0.0479
epoch 12, avg loss 0.0477
epoch 13, avg loss 0.0465
epoch 14, avg loss 0.0454
epoch 15, avg loss 0.0437
epoch 16, avg loss 0.0429
epoch 17, avg loss 0.0405
epoch 18, avg loss 0.0412
epoch 19, avg loss 0.0407
saved trained.pt
